In [ ]:
import xarray as xr

def create_splits(img, input_shape, pad_mode="reflect"):
    # A function to create splits out of a particular size from a given image.
    # images are split up row wise, i.e - row1 split up, row2 split up and so on
    # NOTE - padding is added in case the image can't be split into equal parts
    #       padding is added on the right and the bottom of the image, padding type
    #       is reflected by default
    splits = []

    # calculate pad length  
    pad_len_y = (0 - img.shape[0]) % input_shape[0]  
    pad_len_x = (0 - img.shape[1]) % input_shape[1]

    # option to pad the xarray
    if isinstance(img, xr.DataArray):
        img = img.pad(pad_width={
            "y": (0, pad_len_y), 
            "x": (0, pad_len_x)}, 
            mode=pad_mode
        )

    # option to pad normal numpy array
    else:
        img = np.pad(
            img, 
            [(0, pad_len_y), (0, pad_len_x), (0, 0)], 
            pad_mode
        )

    # loop through the indices with a step of input_shape
    # and create spltis for each index
    for i in range(0, img.shape[0], input_shape[0]):
        for j in range(0, img.shape[1], input_shape[1]):
          # use array slicing to select split size from the whole image
          splits.append(img[i : i + input_shape[0], j : j + input_shape[1]].copy())
    print("Done: create_splits")
    return splits, img


In [4]:
def get_rgb_from_s2(r, g, b):
    # normalize and stack r, g and b bands
    rgb = np.dstack([
        np.squeeze(r),  
        np.squeeze(g),
        np.squeeze(b),
    ])
    normalized_image = rgb.astype(np.float32) / 65535.0

    # clip between 0 and 1 to ensure we 
    # have no data out of range
#     rgb = np.clip(rgb, 0.0, 1.0)
    return rgb

In [3]:
all_tif_10 = glob.glob('/home/vuonghn/research/dataset/satellite/arkansas/org/images/10_*.tif')
# print(all_tif_10)
print(len(all_tif_10))

cdl = glob.glob("/home/vuonghn/research/dataset/satellite/arkansas/org/cdl/*.tif")
print(len(cdl))

NameError: name 'glob' is not defined

In [5]:
import glob
import numpy as np
from skimage import exposure
import os
import rasterio
from typing import List,Dict
import xarray as xr
import cv2
import re
from datetime import datetime

# from spliting import create_splits    

# Define path mappings for different bands
PATH = {
    '10X': '10_',
    '20X': '20_',
    '60X': '60_',
    'SCL': 'SCL_',
}

# Get list of all .tif files in the specified directory
all_tif_10 = glob.glob('/home/vuonghn/research/dataset/satellite/arkansas/org/images/10_*.tif')

def read_cdl(item:str):
    '''
    read cdl
    '''
    path_cdl = "/home/vuonghn/research/dataset/satellite/arkansas/org/cdl/"
    base_name = os.path.basename(item)
    cdl_name = base_name.replace('10_image_','rgb_').replace('.tif','_cdl.tif')
    cdl_path  = os.path.join(path_cdl,cdl_name)
    with rasterio.open(cdl_path) as src:
        image_data = src.read()
        num_bands = src.count
    return image_data

def read_data(path_base:str)->dict:
    """
    Reads and processes raster data for different bands.

    Args:
        path_base (str): The base path of the 10m resolution .tif file.

    Returns:
        dict: A dictionary where keys are band identifiers ('10X', '20X', '60X', 'SCL') and values 
              are tuples containing the image data array, and the x, y coordinates arrays.
    """
    out = {}
    for band in PATH.values():
        base_name = os.path.basename(path_base)
        year,month ,day = base_name.split('_')[-1].split('-')
        day = day.replace('.tif','').replace('(1)','')
        day_of_year = convert_to_day_of_year(int(year),int(month) ,int(day))
        
        print(day)
        base_name = path_base.replace('10_', band)
        print(base_name)
        with rasterio.open(base_name) as src:
            # Read the image data
            image_data = src.read()
            num_bands = src.count
            print(num_bands)
            width = src.width  # Width of the raster in pixels
            height = src.height
            transform = src.transform 
            x, y = np.meshgrid(np.arange(width), np.arange(height))
            x, y = (x * transform[0]) + transform[2], (y * transform[4]) + transform[5]
            out[band] = (image_data, x, y)
    return out,year,day_of_year

def convert_to_day_of_year(year, month, day):
    date = datetime(year, month, day)
    day_of_year = date.timetuple().tm_yday
    return day_of_year

def make_dict(band_dict:dict,cdl):
    '''
    Input:
        Dict X60_, X20_, X10_, SCL_
    Output:
        three Dataframes  
    '''
    img_10 = band_dict['10_'][0]
    _,w,h = img_10.shape
    scl    = band_dict['SCL_'][0]
    scl =np.squeeze(scl)
    upsampled_scl = cv2.resize(
        scl,
        (w,h),
        interpolation=cv2.INTER_NEAREST
    )
    scl = np.moveaxis(upsampled_scl, [0, 1], [1, 0])
    scl = np.expand_dims(scl, axis=0)
    Red, Green, Blue, NIR = img_10[0],img_10[1],img_10[2],img_10[3]  # Band 2 (Red)
    S2_img_10  ={'red':Red,'green':Green,'blue':Blue,'NIR':NIR,'SCL':scl}
    data_stack_10 = np.dstack([
        np.squeeze(S2_img_10['red']),  
        np.squeeze(S2_img_10['green']), 
        np.squeeze(S2_img_10['blue']), 
        np.squeeze(S2_img_10['NIR']), 
        np.squeeze(S2_img_10['SCL']), 
        np.squeeze(cdl),
    ])
     #'B5', 'B6', 'B7', 'B8A', 'B11', 'B12'
    img_20 = band_dict['20_'][0]
    B5, B6, B7 ,B8A, B11,b12 = img_20[0],img_20[1],img_20[2],img_20[3],img_20[4],img_20[5]
    S2_img_20  ={'B5':B5,'B6':B6,'B7':B7,'B8A':B8A,'B11':B11,'B12':b12}
    data_stack_20 = np.dstack([
        np.squeeze(S2_img_20['B5']), 
        np.squeeze(S2_img_20['B6']), 
        np.squeeze(S2_img_20['B7']), 
        np.squeeze(S2_img_20['B8A']), 
        np.squeeze(S2_img_20['B11']),
        np.squeeze(S2_img_20['B12']),
        
    ])
    
    #['B1', 'B9', 'B10']
    img_60  = band_dict['60_'][0]
    B1,B9,B10 = img_60[0], img_60[1],img_60[2]   
    S2_img_60  ={'B1':B1,'B9':B9,'B10':B10}
    data_stack_60 = np.dstack([
        np.squeeze(S2_img_60['B1']), 
        np.squeeze(S2_img_60['B9']), 
        np.squeeze(S2_img_60['B10']),  
    ])
    data_stack_set =[]
    for i, stack in enumerate([data_stack_10, data_stack_20, data_stack_60]):
        print(i)
        if i==0:
            x,y = band_dict['10_'][1],band_dict[
                '10_'][2]
        elif i==1:
            x,y = band_dict['20_'][1],band_dict['20_'][2]
        else:
            x,y = band_dict['60_'][1],band_dict['60_'][2]
        datastack = xr.DataArray(
        stack,
        dims=['y', 'x', 'bands'], 
        coords={
            'y': y[:, 0],  # y coordinates of each pixel
            'x': x[0, :],
            'bands': list(range(stack.shape[2]))

        }
        )
        data_stack_set.append(datastack)
    return data_stack_set
   
        
def splite_data(data_stack_set):
    '''
    tiling
    input:
    output

    '''
    data_stack_xr_10, data_stack_20, data_stack_60  = data_stack_set
    N = 24 #@param {type:"number"}
    splits_10, padded_img_10 = create_splits(data_stack_xr_10, [N, N])
    N = 12 #@param {type:"number"}
    splits_20, padded_img_20 = create_splits(data_stack_20, [N, N])
    N = 4
    splits_60, padded_img_60 = create_splits(data_stack_60, [N, N])


    num_splits_20 = len(splits_20)
    num_splits_60 = len(splits_60)
    print(num_splits_20,num_splits_60)
    return splits_10, splits_20, splits_60

list_tile_10 =[]
list_tile_20 =[]
list_tile_60 =[]
year_list =[]
day_of_year_list =[]


def extract_date(filepath):
    # Extract the date using regex
    match = re.search(r'(\d{4}-\d{2}-\d{2})(?:\(\d+\))?\.tif$', filepath)
    if match:
        date_str = match.group(1)
        return datetime.strptime(date_str, '%Y-%m-%d')
    return datetime.min  # Return minimum date if no match found



sorted_files = sorted(all_tif_10, key=extract_date)

for item in sorted_files:
    # if os. item
    # print("item ", item)
    try:
        print(os.path.basename(item),"*****")
        g,year,day_of_year= read_data(item)
        year_list.append(year)
        day_of_year_list.append(day_of_year)
        cdl = read_cdl(item)
        data_stack_set = make_dict(g,cdl)
        # if "10_image_2023-08-24.tif" in item:
        #     print("zakhire")
        #     data_stack_set1 =data_stack_set


        splits_10, splits_20, splits_60 = splite_data(data_stack_set)
        list_tile_10.append(splits_10)
        list_tile_20.append(splits_20)
        list_tile_60.append(splits_60)
    except Exception as e:
        print(e)
        continue
        
    # print(os.path.basename(item),"*****")
    # g,year,day_of_year= read_data(item)
    # year_list.append(year)
    # day_of_year_list.append(day_of_year)
    # cdl = read_cdl(item)
    # data_stack_set = make_dict(g,cdl)
    # if "10_image_2023-08-24.tif" in item:
    #     print("zakhire")
    #     data_stack_set1 =data_stack_set
        
        
    # splits_10, splits_20, splits_60 = splite_data(data_stack_set)
    # list_tile_10.append(splits_10)
    # list_tile_20.append(splits_20)
    # list_tile_60.append(splits_60)


10_image_2023-01-03.tif *****
03
/home/vuonghn/research/dataset/satellite/arkansas/org/images/10_image_2023-01-03.tif
4
03
/home/vuonghn/research/dataset/satellite/arkansas/org/images/20_image_2023-01-03.tif
6
03
/home/vuonghn/research/dataset/satellite/arkansas/org/images/60_image_2023-01-03.tif
3
03
/home/vuonghn/research/dataset/satellite/arkansas/org/images/SCL_image_2023-01-03.tif
1
/home/vuonghn/research/dataset/satellite/arkansas/org/cdl/rgb_2023-01-03_cdl.tif: No such file or directory
10_image_2023-01-08.tif *****
08
/home/vuonghn/research/dataset/satellite/arkansas/org/images/10_image_2023-01-08.tif
4
08
/home/vuonghn/research/dataset/satellite/arkansas/org/images/20_image_2023-01-08.tif
6
08
/home/vuonghn/research/dataset/satellite/arkansas/org/images/60_image_2023-01-08.tif
/home/vuonghn/research/dataset/satellite/arkansas/org/images/60_image_2023-01-08.tif: No such file or directory
10_image_2023-01-13.tif *****
13
/home/vuonghn/research/dataset/satellite/arkansas/org/imag

In [6]:
numpy_array_10 = np.array(list_tile_10)
numpy_array_20 = np.array(list_tile_20)
numpy_array_60 = np.array(list_tile_60)



In [7]:
print(numpy_array_10.shape)
print(numpy_array_20.shape)
print(numpy_array_60.shape)

(5, 9520, 24, 24, 6)
(5, 9520, 12, 12, 6)
(5, 9520, 4, 4, 3)


In [8]:
len(day_of_year_list)
print(day_of_year_list)

[3, 63, 66, 76, 86, 88]


In [17]:
numpy_array_10.shape

(5, 9520, 24, 24, 6)

In [9]:
a = numpy_array_10.reshape(9520, 5, 24, 24, 6)

In [10]:
series_number, depth, h,w,ch = a.shape

In [11]:
def cloud_cover_check(ds_array):
    # Check for cloudy data in SCL band
    ds_array = np.squeeze(ds_array)
    masked = np.zeros((ds_array.shape[0], ds_array.shape[1]))
    masked[ds_array == 3] = 1
    masked[(ds_array >= 8)] = 1
    percent_valid = (1 - sum(sum(masked)) / (ds_array.shape[0] * ds_array.shape[1])) * 100

    return percent_valid

In [12]:
def nan_perc_check(ds_array):
    # Check for NaN data using the SCL band for S2
    ds_array = np.squeeze(ds_array)
    num_nan = np.isnan(ds_array).astype(int).sum()

    return (num_nan / (ds_array.shape[0] * ds_array.shape[1])) * 100

In [13]:
def filter_date(a,series,dep_lr,tr_shoul):
    id_lst =[]
    cloud =100- cloud_cover_check(a[series][dep_lr,:,:,-2])
    #Red
    value_red = nan_perc_check(a[series][dep_lr,:,:,0])
    if cloud<=10 or value_red<=0:
        pass
    return 
         

In [14]:
def check_data(data,serie_number,dep_lr_number,cloud_tr):
    id_lst =[]
    cloud = 100 - cloud_cover_check(data[serie_number][dep_lr_number,:,:,-2])
    value_red = nan_perc_check(data[serie_number][dep_lr_number,:,:,0])
    print(f'cloud percentage:{cloud}','pixel value:{value_red}')
    if cloud<=cloud_tr and value_red<=0:
        return True
    
    

In [15]:
numpy_array_10 = numpy_array_10.reshape(9520, 5, 24, 24, 6)
numpy_array_20 = numpy_array_20.reshape(9520, 5, 12, 12, 6)
numpy_array_60 = numpy_array_60.reshape(9520, 5, 4, 4, 3)
# (9520, 5, 24, 24, 6)

In [19]:

# series_number = numpy_array_10.shape[1]
import pickle
tr_cloud = 10
tr_miss  = 0
cloud_tr = 0
day = []
year = []
res_10 = []
res_20 = []
res_60 = []

for series in range(series_number):
    for dep_lr in range(depth):
        if check_data(a,series,dep_lr,cloud_tr):
            day.append(day_of_year_list[dep_lr])
            year.append(year_list[dep_lr])
            res_10.append(numpy_array_10[series][dep_lr,:,:,:])
            res_20.append(numpy_array_20[series][dep_lr,:,:,:])
            res_60.append(numpy_array_60[series][dep_lr,:,:,:])

    res_10 =np.array(res_10)
    res_20 =np.array(res_20)
    res_60 =np.array(res_60)
    data = {
    'x10': res_10.astype('int16'),
    'x20': res_20.astype('int16'),
    'x60': res_60.astype('int16'),
    'day': day,
    'year':year,
    'labels': np.squeeze(numpy_array_10[:,1,:,:,4:].astype('int8')),
        }
    with open(f'/home/vuonghn/research/dataset/satellite/arkansas/dataset_vuonghn/{series}.pickle', 'wb') as file:
        pickle.dump(data, file)
    res_10 = []
    res_20 = []
    res_60 = []

            
        
   
        

cloud percentage:0.0 pixel value:{value_red}
cloud percentage:0.0 pixel value:{value_red}
cloud percentage:0.0 pixel value:{value_red}
cloud percentage:0.0 pixel value:{value_red}
cloud percentage:0.0 pixel value:{value_red}
cloud percentage:0.0 pixel value:{value_red}
cloud percentage:0.0 pixel value:{value_red}
cloud percentage:0.0 pixel value:{value_red}
cloud percentage:0.0 pixel value:{value_red}
cloud percentage:0.0 pixel value:{value_red}
cloud percentage:0.0 pixel value:{value_red}
cloud percentage:0.0 pixel value:{value_red}
cloud percentage:0.0 pixel value:{value_red}
cloud percentage:0.0 pixel value:{value_red}
cloud percentage:0.0 pixel value:{value_red}
cloud percentage:0.0 pixel value:{value_red}
cloud percentage:0.0 pixel value:{value_red}
cloud percentage:0.0 pixel value:{value_red}
cloud percentage:0.0 pixel value:{value_red}
cloud percentage:0.0 pixel value:{value_red}
cloud percentage:0.0 pixel value:{value_red}
cloud percentage:0.0 pixel value:{value_red}
cloud perc

In [ ]:
l_10[0][0,:,:,:]

In [ ]:
f.append(a[0][0,:,:,:]).shape

In [ ]:
nan_perc_check(a[0][0,:,:,0])

In [ ]:
numpy_array = np.array(list_tile_10)

In [ ]:
all_tif_10 = glob.glob('/home/vuonghn/research/dataset/satellite/arkansas/org/images/rgb_*.tif')

In [ ]:
all_tif_60 = glob.glob('/home/vuonghn/research/dataset/satellite/arkansas/org/images/TCL_*.tif')

In [ ]:
len(all_tif_10)

In [ ]:
import matplotlib.pyplot as plt 

In [ ]:
sample_split = numpy_array[0][38]
sample = get_rgb_from_s2(
    r=sample_split[:, :, 0], 
    g=sample_split[:, :, 1], 
    b=sample_split[:, :, 2], 
)


fig = plt.figure(figsize = (10, 10))

ax1 = fig.add_subplot(1, 2, 1)
ax1.imshow(sample)
ax1.axis('off');

ax2 = fig.add_subplot(1, 2, 2)
ax2.imshow(sample_split[:, :,5])
ax2.axis('off');

In [ ]:
np.unique(sample_split[:, :,5])

In [ ]:
numpy_array_10[]

In [ ]:
def cloud_cover_check(ds_array):
    # Check for cloudy data in SCL band
    ds_array = np.squeeze(ds_array)

    masked = np.zeros((ds_array.shape[0], ds_array.shape[1]))
    masked[ds_array == 3] = 1
    masked[(ds_array >= 8)] = 1
    percent_valid = (1 - sum(sum(masked)) / (ds_array.shape[0] * ds_array.shape[1])) * 100

    return percent_valid

In [ ]:
def nan_perc_check(ds_array):
    # Check for NaN data using the SCL band for S2
    ds_array = np.squeeze(ds_array)
    num_nan = np.isnan(ds_array).astype(int).sum()
    print("**",num_nan)
    return (num_nan / (ds_array.shape[0] * ds_array.shape[1])) * 100

In [ ]:
cloud_cover_limit = 10 #@param{type:"number"}
RGB_nan_limit = 0 #@param{type:"number"}

filtered_info_df = info_df[info_df['cloud_cover_percentage'] <= cloud_cover_limit]
filtered_info_df = filtered_info_df[filtered_info_df['nan_perc_red'] <= RGB_nan_limit]

print (f'Number of images dropped = {len(info_df) - len(filtered_info_df)} out of {len(info_df)}')

In [ ]:
no_data_check

In [ ]:
non_cloudy_perc = cloud_cover_check(split[:, :, 4])
valid_perc = no_data_check(split[:, :, 4])

In [ ]:
# np.squeeze(S2_img_10['red']),  
# np.squeeze(S2_img_10['green']), 
# np.squeeze(S2_img_10['blue']), 
# np.squeeze(S2_img_10['NIR']), 
# np.squeeze(S2_img_10['SCL']), 
# np.squeeze(cdl),

In [ ]:

N = 24 #@param {type:"number"}
splits_10, padded_img_10 = create_splits(data_stack_set1[0], [N, N])

In [ ]:
for item in splits_10:
    non_cloudy_perc = cloud_cover_check(item[:, :, 4])
#     print(item[:, :, 4].shape)
    valid_perc = no_data_check(item[:, :, 4])
    cloud_cover_perc.append(100 - non_cloudy_perc)
    valid_data_perc.append(valid_perc)    

In [ ]:
np.unique(cloud_cover_perc)

In [ ]:
def no_data_check(ds_array):
    # Check for which data is valid using SCL band for S2
    ds_array = np.squeeze(ds_array)
    valid = (ds_array != 0).astype(int)
    percent_valid = (np.sum(valid) / (ds_array.shape[0] * ds_array.shape[1])) * 100

    return percent_valid

In [ ]:
from collections import Counter
import pandas as pd

In [ ]:
info_df = pd.DataFrame({'index' : list(range(len(numpy_array[0])))})

In [ ]:
# lists to store data for each split
cloud_cover_perc = []
valid_data_perc = []

nan_pix_scl_perc = []
nan_pix_red_perc = []
nan_pix_green_perc = []
nan_pix_blue_perc = []

class_value_count_list = []
class_value_count_abs_list = []

# iterate through the splits
# and perform checks
for split in numpy_array[4]:
    # cloud cover and valid % check using SCL band
    non_cloudy_perc = cloud_cover_check(split[:, :, 4])
    valid_perc = no_data_check(split[:, :, 4])

    # NaN % for each band
    nan_pix_scl_perc.append(nan_perc_check(split[:, :, 4]))
    nan_pix_red_perc.append(nan_perc_check(split[:, :, 0]))
    nan_pix_green_perc.append(nan_perc_check(split[:, :, 1]))
    nan_pix_blue_perc.append(nan_perc_check(split[:, :, 2]))
    
    cloud_cover_perc.append(100 - non_cloudy_perc)
    valid_data_perc.append(valid_perc)    
    
    # Add class distribution for the split
    # this is done by counting the number 
    # of pixels belonging to each class
    # we store the absolute as well as %
    num_pix = split[:, :, 5].shape[0] * split[:, :, 5].shape[1]
    class_value_count = {
        num2class[str(k)] : v / num_pix 
        for k, v in Counter(split[:, :, 5].flatten()).items()
    }
    class_value_count_list.append(class_value_count)

    class_value_count_abs = {
        num2class[str(k)] : v 
        for k, v in Counter(split[:, :, 5].flatten()).items()
    }
    class_value_count_abs_list.append(class_value_count_abs)
  
# create info data frame busing 
# the metrics we computed per split
info_df['cloud_cover_percentage'] = cloud_cover_perc
info_df['valid_data_percentage'] = valid_data_perc
info_df['nan_perc_SCL'] = nan_pix_scl_perc
info_df['nan_perc_red'] = nan_pix_red_perc
info_df['nan_perc_green'] = nan_pix_green_perc
info_df['nan_perc_blue'] = nan_pix_blue_perc
info_df['class_value_count_abs'] = class_value_count_abs_list
info_df['class_value_count'] = class_value_count_list

info_df.head()

In [ ]:
splits = numpy_array[4]

In [ ]:
numpy_array[4].shape

In [ ]:
num_rows = min(len(info_df), 5)
num_cols = 3





fig = plt.figure(figsize = (num_cols * 6, num_rows * 6))
j = 1

# got through the info df and display some images
for i, row in info_df[:num_rows].iterrows():
    s = splits[int(row['index'])]

    # plot the rgb image
    ax = fig.add_subplot(num_rows, num_cols, j)
    ax.imshow(get_rgb_from_s2(
        r=s[:, :, 0], 
        g=s[:, :, 1], 
        b=s[:, :, 2], 
    ));
    ax.yaxis.set_major_locator(plt.NullLocator())
    ax.xaxis.set_major_formatter(plt.NullFormatter())
    ax.set_title('RGB', fontsize = 15)

    # metadata string for the split
    info_string = f"Cloudy percentage: {row['cloud_cover_percentage'] :.02f} %\n\n"
    info_string += f"Valid percentage: {row['valid_data_percentage'] :.02f} %\n\n"
    info_string += f"NaN percentage: {row['nan_perc_red'] :.02f} %\n\n"    
    plt.ylabel(info_string, rotation = 0, fontsize = 15, labelpad = 120, y = 0.27)

    # plot the SCL band
    ax = fig.add_subplot(num_rows, num_cols, j + 1)
    im = ax.imshow(s[:, :, 4], vmax = 11, vmin = 0)
    fig.colorbar(im, fraction=0.046)
    ax.set_title('SCL', fontsize = 15)
    
    ax.yaxis.set_major_locator(plt.NullLocator())
    ax.xaxis.set_major_formatter(plt.NullFormatter())
    
    # plot the EUA mask that we created
    ax = fig.add_subplot(num_rows, num_cols, j + 2)
    im = ax.imshow(s[:, :, 5], vmax = max(num2class.keys()), vmin = min(num2class.keys()))
    fig.colorbar(im, fraction=0.046)
    ax.set_title('European Urban Atlas Mask', fontsize = 15)

    j = j + num_cols
         
plt.suptitle('SPLIT UP IMAGES', fontsize = 20)
fig.subplots_adjust(top=0.97)

In [ ]:
 num2class = {
    "1": "Corn",
    "2": "Cotton",
    "3": "Rice",
    "4": "Sorghum",
    "5": "Soybeans",
    "6": "Sunflower",
    "10": "Peanuts",
    "11": "Tobacco",
    "12": "Sweet Corn",
    "13": "Pop or Orn Corn",
    "14": "Mint",
    "21": "Barley",
    "22": "Durum Wheat",
    "23": "Spring Wheat",
    "24": "Winter Wheat",
    "25": "Other Small Grains",
    "26": "Dbl Crop WinWht/Soybeans",
    "27": "Rye",
    "28": "Oats",
    "29": "Millet",
    "30": "Speltz",
    "31": "Canola",
    "32": "Flaxseed",
    "33": "Safflower",
    "34": "Rape Seed",
    "35": "Mustard",
    "36": "Alfalfa",
    "37": "Other Hay/Non Alfalfa",
    "38": "Camelina",
    "39": "Buckwheat",
    "41": "Sugarbeets",
    "42": "Dry Beans",
    "43": "Potatoes",
    "44": "Other Crops",
    "45": "Sugarcane",
    "46": "Sweet Potatoes",
    "47": "Misc Vegs & Fruits",
    "48": "Watermelons",
    "49": "Onions",
    "50": "Cucumbers",
    "51": "Chick Peas",
    "52": "Lentils",
    "53": "Peas",
    "54": "Tomatoes",
    "55": "Caneberries",
    "56": "Hops",
    "57": "Herbs",
    "58": "Clover/Wildflowers",
    "59": "Sod/Grass Seed",
    "60": "Switchgrass",
    "61": "Fallow/Idle Cropland",
    "62": "Pasture/Grass",
    "63": "Forest",
    "64": "Shrubland",
    "65": "Barren",
    "66": "Cherries",
    "67": "Peaches",
    "68": "Apples",
    "69": "Grapes",
    "70": "Christmas Trees",
    "71": "Other Tree Crops",
    "72": "Citrus",
    "74": "Pecans",
    "75": "Almonds",
    "76": "Walnuts",
    "77": "Pears",
    "81": "Clouds/No Data",
    "82": "Developed",
    "83": "Water",
    "87": "Wetlands",
    "88": "Nonag/Undefined",
    "92": "Aquaculture",
    "111": "Open Water",
    "112": "Perennial Ice/Snow",
    "121": "Developed/Open Space",
    "122": "Developed/Low Intensity",
    "123": "Developed/Med Intensity",
    "124": "Developed/High Intensity",
    "131": "Barren",
    "141": "Deciduous Forest",
    "142": "Evergreen Forest",
    "143": "Mixed Forest",
    "152": "Shrubland",
    "176": "Grassland/Pasture",
    "190": "Woody Wetlands",
    "195": "Herbaceous Wetlands",
    "204": "Pistachios",
    "205": "Triticale",
    "206": "Carrots",
    "207": "Asparagus",
    "208": "Garlic",
    "209": "Cantaloupes",
    "210": "Prunes",
    "211": "Olives",
    "212": "Oranges",
    "213": "Honeydew Melons",
    "214": "Broccoli",
    "215": "Avocados",
    "216": "Peppers",
    "217": "Pomegranates",
    "218": "Nectarines",
    "219": "Greens",
    "220": "Plums",
    "221": "Strawberries",
    "222": "Squash",
    "223": "Apricots",
    "224": "Vetch",
    "225": "Dbl Crop WinWht/Corn",
    "226": "Dbl Crop Oats/Corn",
    "227": "Lettuce",
    "228": "Dbl Crop Triticale/Corn",
    "229": "Pumpkins",
    "230": "Dbl Crop Lettuce/Durum Wht",
    "231": "Dbl Crop Lettuce/Cantaloupe",
    "232": "Dbl Crop Lettuce/Cotton",
    "233": "Dbl Crop Lettuce/Barley",
    "234": "Dbl Crop Durum Wht/Sorghum",
    "235": "Dbl Crop Barley/Sorghum",
    "236": "Dbl Crop WinWht/Sorghum",
    "237": "Dbl Crop Barley/Corn",
    "238": "Dbl Crop WinWht/Cotton",
    "239": "Dbl Crop Soybeans/Cotton",
    "240": "Dbl Crop Soybeans/Oats",
    "241": "Dbl Crop Corn/Soybeans",
    "242": "Blueberries",
    "243": "Cabbage",
    "244": "Cauliflower",
    "245": "Celery",
    "246": "Radishes",
    "247": "Turnips",
    "248": "Eggplants",
    "249": "Gourds",
    "250": "Cranberries",
    "254": "Dbl Crop Barley/Soybeans",
     "255":"other"

}
 

In [ ]:
from matplotlib.colors import Normalize
from skimage import exposure

In [ ]:
def get_rgb_from_s2(r, g, b):
    # normalize and stack r, g and b bands
    rgb = np.dstack([
        np.squeeze(r),  
        np.squeeze(g),
        np.squeeze(b),
    ])
    equalized_img = exposure.equalize_hist(rgb)

    # clip between 0 and 1 to ensure we 
    # have no data out of range
#     rgb = np.clip(rgb, 0.0, 1.0)
    
    return equalized_img

In [ ]:
def draw_grid(img, grid_shape, color=(0, 255, 0), thickness=1):
    # function to draw a grid of 
    # N x N shape on an image
    h, w, _ = img.shape
    rows, cols = grid_shape
    dy, dx = h / rows, w / cols
    
    # draw vertical lines
    for x in np.linspace(start=dx, stop=w-dx, num=cols-1):
        x = int(round(x))
        cv2.line(img, (x, 0), (x, h), color=color, thickness=thickness)

    # draw horizontal lines
    for y in np.linspace(start=dy, stop=h-dy, num=rows-1):
        y = int(round(y))
        cv2.line(img, (0, y), (w, y), color=color, thickness=thickness)

    return img

def get_filled_splits(img, info_df, drop_color, pick_color):
  # function to color patches according 
  # to if they are valid or not
  splits, _ = create_splits(img, input_shape = [N, N])

  splits_new = []
  valid_idxs = list(info_df['index'])

  # iterate through splits and color them
  # based on if they are valid or not
  for i, split in enumerate(splits):
    if i in valid_idxs:
      mask = np.full_like(split, [pick_color], np.uint8)
      splits_new.append(cv2.addWeighted(split, 0.8, mask, 0.2,0))
    else:
      mask = np.full_like(split, [drop_color], np.uint8)
      splits_new.append(cv2.addWeighted(split, 0.8, mask, 0.2,0))

  return splits_new

def stitch(split_images, input_shape, orig_shape):
    # stitch smaller image splits together to form 
    # the larger image they were created from
    border_len_y = (0 - orig_shape[0]) % input_shape[0]  
    border_len_x = (0 - orig_shape[1]) % input_shape[1]

    # create image of 0's 
    # which is equivalent to the shape of the
    # original image + padding done while splitting
    img_stitched = np.zeros((
        orig_shape[0] + border_len_y, 
        orig_shape[1] + border_len_x, orig_shape[2]
    ))
    k = 0

    # iterate through the idxs and 
    # place  splits acc. to index
    for i in range(0, img_stitched.shape[0], input_shape[0]):
        for j in range(0, img_stitched.shape[1], input_shape[1]):
            img_stitched[i : i + input_shape[0], j : j + input_shape[1]] = split_images[k]
            k = k + 1

    # remove padded area from the image
    img_stitched = img_stitched[0 : orig_shape[0], 0 : orig_shape[1]]

    return img_stitched

In [ ]:
cloud_cover_limit = 10 #@param{type:"number"}
RGB_nan_limit = 0 #@param{type:"number"}

filtered_info_df = info_df[info_df['cloud_cover_percentage'] <= cloud_cover_limit]
filtered_info_df = filtered_info_df[filtered_info_df['nan_perc_red'] <= RGB_nan_limit]

print (f'Number of images dropped = {len(info_df) - len(filtered_info_df)} out of {len(info_df)}')

In [ ]:
N=24
import matplotlib.patches as mpatches

In [ ]:
numpy_array[0,:, :, 0].shape

In [ ]:
sample_split.shape

In [ ]:
sample_split =numpy_array[4]

In [ ]:
sample_split = data_stack_set1[0]

In [ ]:
sample_split[:,:, 1].shape

In [ ]:
# image to display for the plot
disp_img = get_rgb_from_s2(
    r=sample_split[:, :, 0], 
    g=sample_split[:, :, 1], 
    b=sample_split[:, :,2], 
)
N=240
# cast all nan valyes to 0
disp_img = np.nan_to_num(disp_img, 0)
disp_img = (disp_img * 255).astype('uint8')

# get colored splits
colored_splits = get_filled_splits(
    disp_img, 
    filtered_info_df, 
    drop_color=[255, 0, 0], 
    pick_color=[0, 255,0]
)
# print("g")
# print(colored_splits)

# create full image from colored splits
disp_img = stitch(
    colored_splits, 
    input_shape=[N, N], 
    orig_shape=disp_img.shape
).astype('uint8')

plt.figure(figsize = (20, 20))

# draw grids on colored image
plt.imshow(draw_grid(
    disp_img.copy(), 
    grid_shape=[disp_img.shape[0] // N, disp_img.shape[1] // N], 
    color=[255, 255,0], 
    thickness=3
))

plt.axis('off')
red_patch = mpatches.Patch(color='red', label='Invalid data', alpha = 0.5)
green_patch = mpatches.Patch(color='green', label='Valid data', alpha = 0.5)
plt.legend(handles=[red_patch, green_patch], fontsize = 15, loc = (1.01, 0.925))
plt.axis('off');